<a href="https://colab.research.google.com/github/Nihat0311/NLP/blob/main/QLoRA_Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch transformers bitsandbytes peft datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.2 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "gpt2"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Model {device} üzərində işləyəcək.")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

Model cuda üzərində işləyəcək.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["c_attn"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


In [ ]:
from datasets import load_dataset

dataset = load_dataset("timdettmers/openassistant-guanaco", split="train")

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")

tokenized_dataset = dataset.map(tokenize_function, batched=True)

README.md:   0%|          | 0.00/395 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


openassistant_best_replies_train.jsonl:   0%|          | 0.00/20.9M [00:00<?, ?B/s]

openassistant_best_replies_eval.jsonl:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9846 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/518 [00:00<?, ? examples/s]

Map:   0%|          | 0/9846 [00:00<?, ? examples/s]

In [ ]:
dataset["train"][0]

{'text': '### Human: Can you please write me small code snippet for React functional component, that has two divs and each div have multiple items inside of them, source of items is an Array, also each item should be drag & drop between two divs ?### Assistant: Here\'s a sample code for a React functional component with two divs that have items inside of them. The items are obtained from an array and each item can be dragged and dropped between the two divs:\n\n\'\'\'javascript\nimport React, { useState } from "react";\n\nconst App = () => {\n  const [list1, setList1] = useState([\n    { id: 1, text: "Item 1" },\n    { id: 2, text: "Item 2" },\n    { id: 3, text: "Item 3" }\n  ]);\n  const [list2, setList2] = useState([\n    { id: 4, text: "Item 4" },\n    { id: 5, text: "Item 5" }\n  ]);\n\n  const onDragStart = (event, source) => {\n    event.dataTransfer.setData("source", source);\n  };\n\n  const onDrop = (event, target) => {\n    const source = event.dataTransfer.getData("source")

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./qlora_decoder_model",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    fp16=True,
    optim="paged_adamw_8bit",
    num_train_epochs=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.624731
20,3.536464
30,3.520848
40,3.426545
50,3.479230
60,3.422315
70,3.223901
80,3.351100
90,3.379741
100,3.317059


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=616, training_loss=3.2037215821154708, metrics={'train_runtime': 749.7575, 'train_samples_per_second': 13.132, 'train_steps_per_second': 0.822, 'total_flos': 2581601524383744.0, 'train_loss': 3.2037215821154708, 'epoch': 1.0})

In [ ]:
import numpy as np
import math

split_dataset = dataset["train"].train_test_split(test_size=0.1)
train_dataset = split_dataset["train"].map(tokenize_function, batched=True)
eval_dataset = split_dataset["test"].map(tokenize_function, batched=True)

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    return {}

training_args = TrainingArguments(
    output_dir="./qlora_decoder_model",
    eval_strategy="epoch",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
)

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

eval_results = trainer.evaluate()

perplexity = math.exp(eval_results['eval_loss'])
print(f"Eval Loss: {eval_results['eval_loss']:.4f}")
print(f"Perplexity (PPL): {perplexity:.4f}")

Map:   0%|          | 0/7974 [00:00<?, ? examples/s]

Map:   0%|          | 0/887 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Epoch
No log,2.957890,0


Eval Loss: 2.9579
Perplexity (PPL): 19.2573


In [ ]:
!pip install evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.4 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=6d09b541a7b89b8b76e1abc32b749b8f7f4a1a7e04924398095a56e928f174b0
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
import evaluate

rouge = evaluate.load("rouge")

predictions = ["Salam, mən süni intellekt köməkçisiyəm.", "Bu gün hava çox gözəldir."]
references = ["Salam! Mən sizə kömək edəcək süni intellektəm.", "Bu gün hava olduqca gözəldir."]

results = rouge.compute(predictions=predictions, references=references)
print(results)

{'rouge1': np.float64(0.8375), 'rouge2': np.float64(0.6180124223602484), 'rougeL': np.float64(0.7175), 'rougeLsum': np.float64(0.7175)}


In [ ]:
model.eval()
prompt = "Human: What is the capital of France?\nAssistant:"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

output_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

print("--- Modelin Cavabı ---")
print(output_text)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


--- Modelin Cavabı ---
Human: What is the capital of France?
Assistant: It is the capital of France, which is a nation of people, who have a common human culture, which includes a history, culture, and language. The capital is located in Paris, in the city of Bordeaux.

In France
